# KingFisher Flex

```{device-card} thermo-fisher-kingfisher-flex
```

Run a local BindIt BDZ directly through PLR with explicit plate and protocol-pause confirmations.
The tested setup is a matching **96DW head, tip comb and plates**, firmware 1.01.0 on macOS.
Hardware checks include mixing, bead collection/release, both drying positions, 37 C heating,
and a reagent-addition pause. Other documented temperature setpoints are accepted but not
individually hardware-tested. Linux and other head/plate formats remain unvalidated.

**Experimental driver:** partial tip-comb attachment may occur without a firmware error.
Command completion does not verify attachment. Inspect pickup and
physical clearances; there is no automatic retry or recovery. See the
[support and operation guide](validation.md).

The first version executes supported exports; general protocol authoring is deferred.
Onboard virtual Start and protocol upload remain unsupported.


## How it communicates

The Flex uses length-prefixed 64-byte HID reports carrying ASCII command lines. Command acknowledgment and execution completion are distinct. The driver continually reads responses, including temperature reports during motion and operator waits. It does not use the Presto XML protocol or its acknowledgment commands.

## Physical setup

Prepare the installed 96DW head, matching tips and supported plates according to the instrument instructions. Close BindIt before connecting PLR. On Linux install PLR with its HID extra, the system HIDAPI library, and device permissions as described in the validation guide. Each hardware cell below is a separate operation to review; do not run the entire notebook unattended.

## Inspect a protocol offline

Set the path to your original BindIt export. Reading and preflight do not open USB. Unknown script forms remain inspectable but cannot execute. Plate labels are not printed tray numbers.

In [ ]:
from pathlib import Path
from pylabrobot.thermo_fisher.kingfisher.flex import KingFisherFlex, read_flex_bdz

protocol = read_flex_bdz(Path("protocol.bdz"))
commands = protocol.preflight()
(protocol.name, protocol.sha256, len(commands), protocol.prompts())

## Connect and identify

This cell accesses real hardware. `setup()` sends only VER; it does not run initialization or acknowledge faults. Specify `serial_number` if more than one instrument is connected.

In [ ]:
flex = KingFisherFlex()
await flex.setup()
(flex.version, flex.serial)

## Initialize explicitly

**This moves the instrument.** Check the instrument and clearances first. Initialization
has been tested on firmware 1.01.0; the observed final orientation is printed tray 1
at processing and tray 4 at loading. Initialization is separate from opening USB.

In [ ]:
await flex.initialize()
flex.processing_slot

## Observe events

Start one consumer before running. Host load/unload/pause events announce an action;
`host_prompt_completed` records its confirmation. Temperature events preserve heater and
ambient readings. They do not certify well-liquid temperature or target attainment.

In [ ]:
import asyncio
import logging

logging.basicConfig(level=logging.INFO)


async def observe_events():
  async for event in flex.events():
    logging.info("%s: %s; prompt=%s", event.kind, event.text, event.prompt)


observer = asyncio.create_task(observe_events())

## Define operator confirmations

This interactive example uses a separate input thread so instrument telemetry continues
while you respond. Returning exactly `True` confirms the action. Any other response stops
streaming and leaves the session uncertain; it does not automatically abort physical motion.
The same handler is passed separately for plate actions and protocol pauses.

In [ ]:
async def confirm_action(prompt):
  print(f"{prompt.action.upper()}: {prompt.label}")
  print(f"Loading tray {prompt.loading_slot}; processing tray {prompt.processing_slot}")
  answer = await asyncio.to_thread(
    input, "Complete the action, clear your hands, then type CONFIRM: "
  )
  return answer.strip() == "CONFIRM"

## Execute the existing protocol

This cell starts motion and may start heating before loading. Complete each requested
action before confirming in Python; do not press physical Start in this host-confirmed mode.
For reagent additions, the prompt includes the plate, reagent and volume per well.
Text mentioning seconds in a pause label does not set a timer.

The await returns only after all script commands and finalization complete. The driver
serializes commands; validate your method's physical behavior and outcome for its intended use.
To use physical Start instead, omit both confirmation callbacks from `run_protocol`.

In [ ]:
result = await flex.run_protocol(
  protocol,
  confirm_prompt=confirm_action,
  confirm_pause=confirm_action,
  operator_timeout=None,
)
result

## Abort an active run when needed

Use `await flex.abort()` from a concurrently running task to request abort while a command is pending. The abort path sends the Feature/ESC abort sequence and awaits ABO END. In a script, keep the run as an `asyncio.Task` if another task must abort it.

Canceling the Python task or interrupting a notebook cell **does not automatically stop the instrument**. Use explicit abort if the connection remains available, or the instrument controls if communication is lost. After any interruption inspect physical positions, call `stop()`, then establish a fresh session. Do not replay the unfinished script automatically.

## Disconnect

`stop()` closes USB and wakes the event observer. It is not a motion-stop operation; use it here after the run has completed or after explicitly handling interruption.

In [ ]:
await flex.stop()
await observer